# 01 - Exploracao de Dados

Objetivo deste notebook:
- explorar todos os datasets brutos do projeto por fonte
- entender estrutura, colunas, tipos, nulos e duplicatas
- identificar a granularidade de cada base
- registrar regras para o notebook 02 de limpeza e tratamento

Padrao de organizacao:
- cada fonte tem sua propria secao
- funcoes comuns ficam no inicio
- nenhuma limpeza definitiva deve acontecer aqui


## 1. Imports, caminhos e configuracao

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PASTA_DATASETS = Path('../datasets')
PASTA_IDH = PASTA_DATASETS / 'idh'
PASTA_CRIMES = PASTA_DATASETS / 'crimes'
PASTA_POPULACAO = PASTA_DATASETS / 'populacao'
PASTA_EDUCACAO = PASTA_DATASETS / 'educacao'

ANO_REFERENCIA_IDH = 2010

CONFIG_CSV_PADRAO = {
    'sep': ';',
    'encoding': 'utf-8-sig',
    'decimal': ',',
}


## 2. Funcoes auxiliares para exploracao

In [ ]:
def listar_csvs(pasta: Path) -> list[Path]:
    if not pasta.exists():
        return []
    return sorted(pasta.glob('*.csv'))


def ler_csv_padrao(caminho: Path) -> pd.DataFrame:
    df = pd.read_csv(caminho, **CONFIG_CSV_PADRAO)
    return df.dropna(axis=1, how='all')


def resumo_dataframe(df: pd.DataFrame, nome_dataset: str) -> None:
    print(f'Dataset: {nome_dataset}')
    print(f'Linhas: {df.shape[0]}')
    print(f'Colunas: {df.shape[1]}')
    print(f'Duplicatas exatas: {df.duplicated().sum()}')
    print('\nColunas:')
    for coluna in df.columns:
        print('-', coluna)


def diagnostico_qualidade(df: pd.DataFrame) -> None:
    print('Tipos de dados:')
    display(df.dtypes.to_frame('tipo'))
    print('Nulos por coluna:')
    display(df.isna().sum().sort_values(ascending=False).to_frame('qtd_nulos'))
    print('Amostra de estatisticas:')
    display(df.describe(include='all').transpose())


## 3. Inventario de arquivos brutos

Esta celula mostra quais CSVs ja existem em cada fonte. Se uma pasta ainda nao existir, ela aparece vazia e sera preenchida depois.


In [ ]:
inventario = {
    'idh': listar_csvs(PASTA_IDH),
    'crimes': listar_csvs(PASTA_CRIMES),
    'populacao': listar_csvs(PASTA_POPULACAO),
    'educacao': listar_csvs(PASTA_EDUCACAO),
}

for fonte, arquivos in inventario.items():
    print(f'{fonte}:')
    for arquivo in arquivos:
        print(f'  - {arquivo}')
    if not arquivos:
        print('  - nenhum CSV encontrado')


# Fonte 1 - IDH Municipal 2010

## 4.1 Carregar IDH

In [ ]:
arquivos_idh = listar_csvs(PASTA_IDH)
arquivo_idh = arquivos_idh[0]
arquivo_idh


In [ ]:
df_idh_raw = ler_csv_padrao(arquivo_idh)
df_idh_raw.head()


## 4.2 Resumo estrutural do IDH

In [ ]:
resumo_dataframe(df_idh_raw, 'IDH municipal 2010')


In [ ]:
diagnostico_qualidade(df_idh_raw)


## 4.3 Inspecao da territorialidade no IDH

In [ ]:
coluna_territorialidade_idh = next(
    coluna for coluna in df_idh_raw.columns
    if coluna.strip().lower().startswith('territorialidade')
)

coluna_territorialidade_idh


In [ ]:
amostra_territorialidade_idh = pd.DataFrame({
    'territorialidade': df_idh_raw[coluna_territorialidade_idh],
    'nome_municipio_extraido': df_idh_raw[coluna_territorialidade_idh].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True),
    'sigla_uf_extraida': df_idh_raw[coluna_territorialidade_idh].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0],
})

display(amostra_territorialidade_idh.head(20))


In [ ]:
df_chave_idh = df_idh_raw.copy()
df_chave_idh['ano'] = ANO_REFERENCIA_IDH

duplicatas_chave_idh = df_chave_idh.duplicated(subset=[coluna_territorialidade_idh, 'ano']).sum()
print(f'Duplicatas na chave temporaria territorialidade + ano: {duplicatas_chave_idh}')


## 4.4 Conclusoes preliminares do IDH

Pontos para levar ao notebook 02:
- ler como CSV com separador `;`, encoding `utf-8-sig` e decimal `,`
- remover colunas totalmente vazias
- renomear colunas para pt-br padronizado
- criar `ano = 2010`
- extrair `nome_municipio` e `sigla_uf`
- criar `nome_municipio_padronizado`
- preparar `codigo_municipio` por cruzamento com tabela auxiliar


# Fonte 2 - Crimes

## 5.1 Carregar crimes

In [ ]:
arquivos_crimes = listar_csvs(PASTA_CRIMES)
arquivos_crimes


In [ ]:
if arquivos_crimes:
    arquivo_crimes = arquivos_crimes[0]
    df_crimes_raw = ler_csv_padrao(arquivo_crimes)
    display(df_crimes_raw.head())
else:
    print('Nenhum CSV de crimes encontrado em datasets/crimes.')


## 5.2 Resumo estrutural de crimes

In [ ]:
if arquivos_crimes:
    resumo_dataframe(df_crimes_raw, 'crimes')
    diagnostico_qualidade(df_crimes_raw)


## 5.3 Pontos a decidir para crimes

Registrar aqui depois da exploracao:
- coluna de `codigo_municipio`
- coluna de `ano`
- coluna de quantidade de ocorrencias
- categorias de crime que serao usadas
- regra de agregacao para `municipio + ano`


# Fonte 3 - Populacao

In [ ]:
arquivos_populacao = listar_csvs(PASTA_POPULACAO)
if arquivos_populacao:
    arquivo_populacao = arquivos_populacao[0]
    df_populacao_raw = ler_csv_padrao(arquivo_populacao)
    display(df_populacao_raw.head())
    resumo_dataframe(df_populacao_raw, 'populacao')
    diagnostico_qualidade(df_populacao_raw)
else:
    print('Nenhum CSV de populacao encontrado em datasets/populacao.')


# Fonte 4 - Educacao

In [ ]:
arquivos_educacao = listar_csvs(PASTA_EDUCACAO)
if arquivos_educacao:
    arquivo_educacao = arquivos_educacao[0]
    df_educacao_raw = ler_csv_padrao(arquivo_educacao)
    display(df_educacao_raw.head())
    resumo_dataframe(df_educacao_raw, 'educacao')
    diagnostico_qualidade(df_educacao_raw)
else:
    print('Nenhum CSV de educacao encontrado em datasets/educacao.')


# Conclusao geral da exploracao

Ao final deste notebook, cada fonte deve ter uma lista clara de regras para o notebook 02:
- quais colunas serao mantidas
- quais colunas precisam ser renomeadas
- quais chaves serao usadas
- qual granularidade precisa ser atingida
- quais problemas de qualidade precisam ser tratados
